In [1]:
# Linear Regression exercises using Shotput dataset from UIUC Stat 425 Statistical Modelling (uses R)
# Goal: predict shot.putt (best shot put in meters) from power.clean (beset 1 rep max power clean in kg)
# File created: 25 November 2026
# Last edited: 25 November 2026

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

np.set_printoptions(suppress = True)
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("Datasets/Shotput.csv")
data.head()

,power.clean,shot.putt
0,37.5,6.4
1,51.5,10.2
2,61.3,12.4
3,61.3,13.0
4,63.6,13.2


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   power.clean  28 non-null     float64
 1   shot.putt    28 non-null     float64
dtypes: float64(2)
memory usage: 576.0 bytes


In [5]:
data.columns

Index(['power.clean', 'shot.putt'], dtype='object')

In [6]:
# Predicting "shot.putt" from "power.clean" using sklearn.linear_model's LinearRegression - Stat 425 HW1
print("Simple Linear Regression: shot.putt = b0 + b1 x power.clean + eps")
Y = data["shot.putt"].to_numpy()
X = data["power.clean"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

Simple Linear Regression: shot.putt = b0 + b1 x power.clean + eps
(28, 1) (28,)
beta0 = 5.9596290983739575, beta1 = 0.09834382159781993
Rsquare = 0.7974345503917665


In [7]:
class Jie_SimpleLinearRegression:
    
    def __init__(self):
        self.beta0 = None
        self.beta1 = None
        self.sigma2 = None
        self.Rsquare = None
        self.beta0_stdev = None
        self.beta1_stdev = None
        self.beta0_t = None
        self.beta1_t = None
        self.beta0_p = None
        self.beta1_p = None
        self.results = None
    
    def fit(self, X, Y, predictor_name):
        
        Xc = np.insert(X, 0, np.ones(len(X)), axis = 1)
        self.beta0, self.beta1 = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ Y
        
        Yhat = self.predict(X)
        self.sigma2 = (np.linalg.norm(Y - Yhat) ** 2) / (len(Y) - 2)
        
        beta_var = np.diag(self.sigma2 * np.linalg.inv(Xc.T @ Xc))
        self.beta0_stdev, self.beta1_stdev = np.sqrt(beta_var)

        self.beta0_t = self.beta0 / self.beta0_stdev
        self.beta1_t = self.beta1 / self.beta1_stdev
        
        self.beta0_p = 2 * norm.sf(np.abs(self.beta0_t))
        self.beta1_p = 2 * norm.sf(np.abs(self.beta1_t))

        beta = np.array([self.beta0, self.beta1])
        beta_stdev = np.array([self.beta0_stdev, self.beta1_stdev])
        beta_t = np.array([self.beta0_t, self.beta1_t])
        beta_p = np.array([self.beta0_p, self.beta1_p])
        self.results = pd.DataFrame(zip(beta, beta_stdev, beta_t, beta_p),
                                    columns = ["beta", "beta stdev", "beta t", "beta p"],
                                    index = ["intercept", predictor_name])
        print(self.results)
        
    def score(self, X, Y):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = self.predict(X)
        RSS = np.linalg.norm(Y - Yhat) ** 2
        TSS = np.linalg.norm(Y - np.mean(Y)) ** 2
        self.Rsquare = 1 - RSS/TSS
        print("Rsquare = %s" % self.Rsquare)
        
    def predict(self, X):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = (self.beta0 + self.beta1 * X).reshape(-1)
        return Yhat

In [8]:
# Predicting "shot.putt" from "power.clean" by hand - Stat 425 HW1
print("Simple Linear Regression: shot.putt = b0 + b1 x power.clean + eps")
Y = data["shot.putt"].to_numpy()
predictor_name = "power.clean"
X = data[predictor_name].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = Jie_SimpleLinearRegression()
model.fit(X, Y, predictor_name)
model.score(X, Y)

Simple Linear Regression: shot.putt = b0 + b1 x power.clean + eps
(28, 1) (28,)
                 beta  beta stdev     beta t        beta p
intercept    5.959629    0.958835   6.215493  5.116391e-10
power.clean  0.098344    0.009721  10.116994  4.644593e-24
Rsquare = 0.7974345503917667
